# Harry Potter Knowledge Graph — Projet de webdatamining

## Installation & Imports des bibliothèques


In [1]:
!pip install rdflib requests spacy owlready2 pykeen torch \
    scikit-learn matplotlib seaborn pandas numpy tqdm
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 12.8/12.8 MB 92.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, re, json, time, logging, warnings
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS, OWL, XSD
from rdflib.namespace import SKOS

import spacy
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

# Création de l'arborescence projet
for d in ['data/samples', 'kg_artifacts', 'reports', 'models', 'src']:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


## Data Acquisition via le Wikidata sur Harry Potter


In [4]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "HPProject/1.0"}

QUERIES = {
    "series": """
        SELECT DISTINCT ?char ?charLabel ?genderLabel WHERE {
          ?book wdt:P179 wd:Q8337 .
          ?char wdt:P1441 ?book .
          OPTIONAL { ?char wdt:P21 ?gender }
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
        } LIMIT 300
    """,
    "hogwarts": """
        SELECT DISTINCT ?char ?charLabel ?genderLabel WHERE {
          ?char wdt:P69 wd:Q25975 .
          OPTIONAL { ?char wdt:P21 ?gender }
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
        } LIMIT 300
    """,
    "universe": """
        SELECT DISTINCT ?char ?charLabel ?genderLabel WHERE {
          ?char wdt:P1080 wd:Q5410773 .
          OPTIONAL { ?char wdt:P21 ?gender }
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
        } LIMIT 300
    """,
    "manual": """
        SELECT DISTINCT ?char ?charLabel ?genderLabel WHERE {
          VALUES ?char {
            wd:Q3244512 wd:Q180659 wd:Q184805 wd:Q170687 wd:Q174904
            wd:Q174921 wd:Q217246 wd:Q229054 wd:Q236306 wd:Q260819
            wd:Q264983 wd:Q273930 wd:Q274466 wd:Q294927 wd:Q312965
            wd:Q313226 wd:Q355566 wd:Q359457 wd:Q382274 wd:Q392049
            wd:Q435651 wd:Q464118 wd:Q700705 wd:Q734209 wd:Q850756
            wd:Q866225 wd:Q936765 wd:Q975871 wd:Q1043120 wd:Q1124106
            wd:Q1183173 wd:Q1230184 wd:Q1397671 wd:Q1421901 wd:Q1525851
            wd:Q1574844 wd:Q1637868 wd:Q1682310 wd:Q1770841 wd:Q1972781
            wd:Q2099744 wd:Q2265598 wd:Q2380867 wd:Q2674728 wd:Q2737248
            wd:Q3221958 wd:Q3271644 wd:Q4973503 wd:Q5256612 wd:Q7374115
          }
          OPTIONAL { ?char wdt:P21 ?gender }
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
        }
    """,
}

rows = {}
for name, q in QUERIES.items():
    try:
        resp = requests.get(WIKIDATA_SPARQL,
                            params={"query": q, "format": "json"},
                            headers=HEADERS, timeout=60)
        bindings = resp.json()["results"]["bindings"]
        count = 0
        for b in bindings:
            qid    = b["char"]["value"].split("/")[-1]
            label  = b.get("charLabel", {}).get("value", "")
            gender = b.get("genderLabel", {}).get("value", "")
            if label and not label.startswith("Q") and qid not in rows:
                rows[qid] = {"wd_id": qid, "label": label, "gender": gender}
                count += 1
        print(f"  [{name}] +{count} personnages")
    except Exception as e:
        print(f"  [{name}] erreur : {e}")

df_chars = pd.DataFrame(list(rows.values()))
print(f"\n df_chars total : {len(df_chars)} personnages")
print(df_chars[["label","wd_id","gender"]].head(15))

  [series] +78 personnages
  [hogwarts] +0 personnages
  [universe] +259 personnages
  [manual] +39 personnages

 df_chars total : 376 personnages
                   label       wd_id  gender
0       Albus Dumbledore     Q712548    male
1                  kappa     Q335140        
2           Harry Potter    Q3244512    male
3   Hugo Granger-Weasley    Q3741059    male
4         Barnabas Cuffe   Q21502388    male
5          Eldred Worple  Q106392724    male
6       Dolores Umbridge     Q716941  female
7   Rose Granger-Weasley    Q3744404  female
8       Lily Luna Potter    Q3745681  female
9           Sirius Black     Q713701    male
10         Shell Cottage    Q2793775        
11    Battle of Hogwarts    Q1251278        
12          Amos Diggory    Q1884607    male
13           Viktor Krum    Q1251394    male
14         Devil's Snare    Q1472233        


## Creation of the different classes

In [ ]:
def sparql(query, timeout=60):
    try:
        r = requests.get(WIKIDATA_SPARQL,
                         params={"query": query, "format": "json"},
                         headers=HEADERS, timeout=timeout)
        return r.json()["results"]["bindings"]
    except Exception as e:
        print(f"erreur SPARQL : {e}")
        return []

def val(row, key):
    return row.get(key, {}).get("value", None)

def qid(uri):
    return uri.split("/")[-1] if uri and uri.startswith("http") else uri

# Livres
Q_BOOKS = """
SELECT DISTINCT ?item ?itemLabel ?pubDate ?pages WHERE {
  ?item wdt:P179 wd:Q8337 .
  OPTIONAL { ?item wdt:P577 ?pubDate }
  OPTIONAL { ?item wdt:P1104 ?pages }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
"""
raw_books = sparql(Q_BOOKS)
df_books = pd.DataFrame([{
    "wd_id" : qid(val(r, "item")),
    "label" : val(r, "itemLabel"),
    "pubDate": val(r, "pubDate"),
    "pages" : val(r, "pages"),
} for r in raw_books]).drop_duplicates("wd_id")

# Famille
REL_LABELS = {"P22":"hasFather","P25":"hasMother","P26":"hasSpouse",
              "P3373":"hasSibling","P40":"hasChild"}
Q_FAMILY = """
SELECT DISTINCT ?person ?personLabel ?relType ?target ?targetLabel WHERE {
  VALUES ?relType { wdt:P22 wdt:P25 wdt:P26 wdt:P3373 wdt:P40 }
  ?person ?relType ?target .
  ?person wdt:P1080 wd:Q5410773 .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
} LIMIT 2000
"""
raw_family = sparql(Q_FAMILY)
df_family = pd.DataFrame([{
    "subject"      : qid(val(r, "person")),
    "subject_label": val(r, "personLabel"),
    "relation"     : REL_LABELS.get(val(r, "relType",).split("/")[-1] if val(r,"relType") else "", "relatedTo"),
    "object"       : qid(val(r, "target")),
    "object_label" : val(r, "targetLabel"),
} for r in raw_family]).drop_duplicates()

# Maisons
Q_HOUSES = """
SELECT DISTINCT ?item ?itemLabel ?founderLabel WHERE {
  VALUES ?item { wd:Q170534 wd:Q170547 wd:Q170548 wd:Q170546 }
  OPTIONAL { ?item wdt:P112 ?founder }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
"""
raw_houses = sparql(Q_HOUSES)
df_houses = pd.DataFrame([{
    "wd_id"  : qid(val(r, "item")),
    "label"  : val(r, "itemLabel"),
    "founder": val(r, "founderLabel"),
} for r in raw_houses]).drop_duplicates("wd_id")

# Appartenances aux maisons 
Q_MEMBERSHIPS = """
SELECT DISTINCT ?person ?personLabel ?houseLabel WHERE {
  VALUES ?house { wd:Q170534 wd:Q170547 wd:Q170548 wd:Q170546 }
  { ?person wdt:P361 ?house } UNION { ?person wdt:P1080 wd:Q5410773 . ?person wdt:P361 ?house }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
} LIMIT 1000
"""
raw_memberships = sparql(Q_MEMBERSHIPS)
df_memberships = pd.DataFrame([{
    "person_id"   : qid(val(r, "person")),
    "person_label": val(r, "personLabel"),
    "house_label" : val(r, "houseLabel"),
} for r in raw_memberships]).drop_duplicates()

print(f"DataFrames construits")
print(f"  Personnages : {len(df_chars)}")
print(f"  Livres      : {len(df_books)}")
print(f"  Relations   : {len(df_family)}")
print(f"  Maisons     : {len(df_houses)}")
print(f"  Appartenances: {len(df_memberships)}")


DataFrames construits
  Personnages : 376
  Livres      : 10
  Relations   : 518
  Maisons     : 4
  Appartenances: 0


## NER (Named Entity Recognition) with spaCy

In [ ]:
nlp = spacy.load("en_core_web_sm")

TEXTS = [
    "Harry Potter is the main character. He attends Hogwarts School of Witchcraft and Wizardry.",
    "Albus Dumbledore is the headmaster of Hogwarts and leads the Order of the Phoenix.",
    "Hermione Granger, Ron Weasley and Harry Potter are best friends in Gryffindor house.",
    "Lord Voldemort, also known as Tom Riddle, is the main antagonist of the series.",
    "Professor Severus Snape teaches Potions at Hogwarts and is head of Slytherin.",
    "Draco Malfoy is a student in Slytherin and Harry Potter's rival.",
    "Sirius Black escaped from Azkaban prison and is Harry Potter's godfather.",
    "Rubeus Hagrid is the Keeper of Keys and Grounds at Hogwarts.",
    "Bellatrix Lestrange is a Death Eater and loyal follower of Voldemort.",
    "Neville Longbottom becomes an important member of Dumbledore's Army.",
]

all_entities = []
for text in TEXTS:
    doc = nlp(text)
    for ent in doc.ents:
        all_entities.append({
            "text" : ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end"  : ent.end_char,
            "context": text[:80]
        })

df_ner = pd.DataFrame(all_entities)
print(f"{len(df_ner)} entités extraites")
print(df_ner.groupby("label")["text"].apply(list))

27 entités extraites
label
FAC            [the Keeper of Keys and Grounds at Hogwarts]
GPE              [Phoenix, Gryffindor, Neville, Longbottom]
LOC                                             [Voldemort]
NORP                                              [Azkaban]
ORG       [Hogwarts School of Witchcraft, Wizardry, Albu...
PERSON    [Harry Potter, Hogwarts, Hermione Granger, Ron...
Name: text, dtype: object
